# projector_representation

已訓練 SimCLR → 🔒 Freeze → **Projector輸出(128維)** → Linear Probe → Accuracy

*拿同一個 Baseline 模型換一種方式考試*

## step1:Import + Dataset

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18

import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

linear_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=linear_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=linear_transform
)

batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


c:\Users\user\anaconda3\envs\summer\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000
Test: 10000


## step2:建立與 Experiment 1 相同的 SimCLR Model

In [2]:
class SimCLRModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = resnet18(weights=None)

        self.backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Identity()

        self.projector = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.projector(h)
        return h, z

## step3:載入 Experiment 1 的 Baseline 權重

In [3]:
model = SimCLRModel().to(device)

model.load_state_dict(
    torch.load(
        "simclr_baseline_final.pth",
        map_location=device
    )
)

model.eval()

print("Baseline SimCLR model loaded.")

Baseline SimCLR model loaded.


## step4:Freeze Backbone + Projector

In [4]:
for param in model.parameters():
    param.requires_grad = False

model.eval()

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Trainable SimCLR parameters:", trainable_params)

Trainable SimCLR parameters: 0


## step5:建立 128 → 10 Linear Classifier

In [5]:
linear_classifier = nn.Linear(128, 10).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    linear_classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-6
)

print(linear_classifier)

Linear(in_features=128, out_features=10, bias=True)


## step6:正式 Linear Probing 100 Epochs

In [6]:
num_epochs = 100

loss_history = []
test_accuracy_history = []

start_time = time.time()

for epoch in range(1, num_epochs + 1):

    # =====================
    # Train Linear Layer
    # =====================
    linear_classifier.train()
    model.eval()

    total_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.no_grad():
            _, projector_features = model(images)

        outputs = linear_classifier(projector_features)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    # =====================
    # Test
    # =====================
    linear_classifier.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            _, projector_features = model(images)

            outputs = linear_classifier(projector_features)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    test_acc = 100.0 * correct / total
    test_accuracy_history.append(test_acc)

    print(
        f"Epoch [{epoch:3d}/{num_epochs}] "
        f"| Loss: {avg_loss:.4f} "
        f"| Test Accuracy: {test_acc:.2f}%"
    )


elapsed = time.time() - start_time

print("\n===== Projector Representation Finished =====")
print(f"Training Time: {elapsed/60:.2f} minutes")
print(f"Final Test Accuracy: {test_accuracy_history[-1]:.2f}%")
print(f"Best Test Accuracy: {max(test_accuracy_history):.2f}%")

Epoch [  1/100] | Loss: 0.7654 | Test Accuracy: 82.40%
Epoch [  2/100] | Loss: 0.4844 | Test Accuracy: 82.59%
Epoch [  3/100] | Loss: 0.4695 | Test Accuracy: 82.87%
Epoch [  4/100] | Loss: 0.4634 | Test Accuracy: 83.01%
Epoch [  5/100] | Loss: 0.4609 | Test Accuracy: 83.14%
Epoch [  6/100] | Loss: 0.4589 | Test Accuracy: 83.01%
Epoch [  7/100] | Loss: 0.4569 | Test Accuracy: 82.99%
Epoch [  8/100] | Loss: 0.4561 | Test Accuracy: 82.99%
Epoch [  9/100] | Loss: 0.4560 | Test Accuracy: 82.80%
Epoch [ 10/100] | Loss: 0.4545 | Test Accuracy: 83.12%
Epoch [ 11/100] | Loss: 0.4532 | Test Accuracy: 82.80%
Epoch [ 12/100] | Loss: 0.4534 | Test Accuracy: 83.26%
Epoch [ 13/100] | Loss: 0.4537 | Test Accuracy: 83.17%
Epoch [ 14/100] | Loss: 0.4527 | Test Accuracy: 82.93%
Epoch [ 15/100] | Loss: 0.4526 | Test Accuracy: 82.91%
Epoch [ 16/100] | Loss: 0.4519 | Test Accuracy: 83.30%
Epoch [ 17/100] | Loss: 0.4527 | Test Accuracy: 82.93%
Epoch [ 18/100] | Loss: 0.4519 | Test Accuracy: 83.09%
Epoch [ 19